In [1]:
import pandas as pd, json
from pathlib import Path

In [2]:
github_csv = pd.read_csv("projects.csv", dtype=str).fillna("")
github_csv["name_norm"] = github_csv["name"].str.lower().str.strip()

github_projects = [
    {
        "name": r["name"],
        "name_norm": r["name_norm"],
        "visibility": r.get("visibility", ""),
        "language": r.get("language", ""),
        "license": r.get("license", ""),
        "updated": r.get("updatedAt", "")
    }
    for _, r in github_csv.iterrows()
]

github_map = {p["name_norm"]: p for p in github_projects}


data = json.load(open("projects.json", "r", encoding="utf-8"))
prime = data.get("prime", {})

registry_projects = []
for category, items in prime.items():
    for it in items:
        ent = dict(it)
        ent["category"] = category
        ent["name_norm"] = ent.get("name", "").lower().strip()
        registry_projects.append(ent)

registry_map = {p["name_norm"]: p for p in registry_projects}

In [3]:
rows = []
registry_names = set(registry_map.keys())
github_names = set(github_map.keys())

for name_norm, reg in registry_map.items():
    gh = github_map.get(name_norm)  

    rows.append({
        "updated": gh["updated"] if gh else "",
        "category": reg.get("category"),
        "registry_name": reg.get("name"),
        "language": gh["language"] if gh else "",
        "status": reg.get("status", ""),

        "visibility": gh["visibility"] if gh else "",
        "github_name": gh["name"] if gh else "",
        "github_present": gh is not None,
    })

df = (
    pd.DataFrame(rows)
    .sort_values(["category", "registry_name"])
    .reset_index(drop=True)
)

matched = df[df["github_present"]]
only_registry = df[~df["github_present"]]
only_github = sorted(github_names - registry_names)

print(f"Registry JSON total: {len(registry_projects)}")
print(f"GitHub CSV total: {len(github_projects)}")
print(f"Matched (in both): {len(matched)}")
print(f"Only in registry JSON: {len(only_registry)}")
print(f"Only in GitHub CSV: {len(only_github)}")

Registry JSON total: 47
GitHub CSV total: 48
Matched (in both): 47
Only in registry JSON: 0
Only in GitHub CSV: 1


In [ ]:
if only_github:
    print("Projects in GitHub CSV not found in registry JSON:")
    display(pd.DataFrame({"name_norm": only_github}))


if not only_registry.empty:
    print("Projects in registry JSON but missing in GitHub CSV:")
    display(only_registry[["registry_name", "category"]])

Projects in GitHub CSV not found in registry JSON (normalized names):


,name_norm
0,nma


In [5]:
display(df)

,updated,category,registry_name,language,status,visibility,github_name,github_present
0,2026-02-10 09:46:48,Data,sparklineo,,archived,PRIVATE,sparklineo,True
1,2026-02-28 07:08:59,Games,echess,,failed,PUBLIC,echess,True
2,2026-03-01 13:06:44,Games,games.py,,active,PUBLIC,games.py,True
3,2026-02-24 10:39:56,Games,games.rs,,archived,PUBLIC,games.rs,True
4,2026-02-27 08:20:38,Games,games.ts,,active,PUBLIC,games.ts,True
5,2026-02-12 09:30:49,Games,wordle,,archived,PUBLIC,wordle,True
6,2025-04-03 19:06:17,Interactive,astronomy,,successful,PUBLIC,astronomy,True
7,2026-02-28 09:14:19,Interactive,mash3d,,successful,PRIVATE,mash3d,True
8,2026-02-28 06:50:32,Interactive,moon3d,,archived,PRIVATE,moon3d,True
9,2026-02-28 06:40:19,LLM,lessontube,,failed,PUBLIC,lessontube,True
